In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")


In [ ]:
possible_paths = [Path("Dataset/car_data_original.csv"), Path("../Dataset/car_data_original.csv"), Path("car_data_original.csv"), Path("/mnt/data/car data.csv")]
data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Place car_data_original.csv in the Dataset folder.")
raw_data = pd.read_csv(data_path)
raw_data.head()


In [ ]:
print("Dataset shape:", raw_data.shape)
print("\nData types:")
print(raw_data.dtypes)
print("\nMissing values:")
print(raw_data.isnull().sum())
print("\nDuplicate rows:", raw_data.duplicated().sum())
raw_data.describe(include="all")


In [ ]:
data = raw_data.copy()
data.columns = [column.strip().replace(" ", "_") for column in data.columns]
for column in data.select_dtypes(include="object").columns:
    data[column] = data[column].astype(str).str.strip()
duplicates_removed = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", duplicates_removed)
print("Cleaned shape:", data.shape)
print("Missing values after cleaning:", int(data.isnull().sum().sum()))


In [ ]:
cleaned_output = Path("Dataset/car_data_cleaned.csv")
if not cleaned_output.parent.exists():
    cleaned_output = Path("../Dataset/car_data_cleaned.csv")
if cleaned_output.parent.exists():
    data.to_csv(cleaned_output, index=False)


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data["Selling_Price"], bins=25, kde=True)
plt.title("Distribution of Car Selling Prices")
plt.xlabel("Selling Price")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="Present_Price", y="Selling_Price", hue="Fuel_Type")
plt.title("Selling Price vs Present Price")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="Fuel_Type", y="Selling_Price")
plt.title("Selling Price Distribution by Fuel Type")
plt.tight_layout()
plt.show()


In [ ]:
model_data = data.copy()
model_data["Vehicle_Age"] = model_data["Year"].max() - model_data["Year"]
target = "Selling_Price"
features = ["Car_Name", "Vehicle_Age", "Present_Price", "Driven_kms", "Fuel_Type", "Selling_type", "Transmission", "Owner"]
X = model_data[features]
y = model_data[target]
numeric_features = ["Vehicle_Age", "Present_Price", "Driven_kms", "Owner"]
categorical_features = ["Car_Name", "Fuel_Type", "Selling_type", "Transmission"]


In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=400, random_state=42, min_samples_leaf=1)
}
model_results = {}
trained_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    trained_models[name] = (pipeline, predictions)
    model_results[name] = {
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "R2": r2_score(y_test, predictions)
    }
metrics_df = pd.DataFrame(model_results).T.sort_values("R2", ascending=False)
metrics_df


In [ ]:
best_model_name = metrics_df.index[0]
best_pipeline, best_predictions = trained_models[best_model_name]
print("Selected model:", best_model_name)
print("MAE:", mean_absolute_error(y_test, best_predictions))
print("RMSE:", mean_squared_error(y_test, best_predictions) ** 0.5)
print("R2:", r2_score(y_test, best_predictions))


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.75)
limits = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(limits, limits, linestyle="--")
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title(f"Actual vs Predicted Selling Price: {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
permutation = permutation_importance(best_pipeline, X_test, y_test, n_repeats=15, random_state=42, scoring="r2")
importance_df = pd.DataFrame({"Feature": features, "Importance": permutation.importances_mean}).sort_values("Importance", ascending=False)
importance_df


In [ ]:
predictions_output = pd.DataFrame({"Actual_Selling_Price": y_test.values, "Predicted_Selling_Price": best_predictions})
predictions_output.head(10)
